In [ ]:
import pandas as pd
# On recharge le fichier CSV généré à l'étape précédente
df = pd.read_csv("churn_data.csv")
df.head() # affiche les 5 premières lignes, pour un contrôle visuel rapide

In [ ]:
df.shape # (nombre de lignes, nombre de colonnes) -> doit afficher (500, 6)

In [ ]:
df.describe() # count (500 clients), moyenne, min, max, écart-type, 25%, 50%, 75% desclients

In [ ]:
df.isnull().sum() # compte les valeurs manquantes par colonne -> les 500 lignes sontcomplètes, aucune donnée n'est vide, doit être 0 partout

In [ ]:
df["churn"].value_counts() # compte les clients "0" (restent) vs "1" (partent)

In [ ]:
from sklearn.model_selection import train_test_split
# ============================================================
# 1. Conversion de la colonne « abonnement »
# ============================================================
# Les algorithmes de ML ne comprennent que des nombres : on convertit
# le texte de la colonne "abonnement" en valeurs numériques.
df["abonnement"] = df["abonnement"].map({
"Basique": 0,
"Standard": 1,
"Premium": 2
})
# ============================================================
# 2. Définition des variables explicatives (features)
# ============================================================
X = df[
[
"age",
"salaire",
"abonnement",
"anciennete",
"consommation"
]
]
# ============================================================
# 3. Définition de la variable cible (target)
# ============================================================
y = df["churn"]
# ============================================================
# 4. Séparation Train / Test
# - 80% pour entraîner le modèle (X_train, y_train)
# - 20% mis de côté, jamais vus pendant l'entraînement,
# pour évaluer honnêtement le modèle (X_test, y_test)
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
X,
y,
test_size=0.2, # 20% des données réservées au test
random_state=42 # même logique de reproductibilité que pour le dataset

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
# On crée le modèle : une forêt de 100 arbres de décision qui votent ensemble.
# max_depth=5 limite la complexité de chaque arbre (évite qu'il "apprenne par cœur").
model = RandomForestClassifier(n_estimators=100,
max_depth=5,
random_state=42
)
# .fit() est le mot qui déclenche l'APPRENTISSAGE proprement dit :
# le modèle analyse X_train et y_train pour trouver une relation entre les deux.
# La cellule qui affiche « RandomForestClassifier(...) » indique simplement
# que le modèle a été entraîné avec succès. À ce stade, aucune prédiction
# n'a encore été réalisée : le modèle a appris à partir des 400 clients
# d'entraînement et est maintenant prêt à être testé sur les 100 restants.
model.fit(X_train, y_train)
# On demande au modèle de prédire sur les données de test (jamais vues avant).
y_pred = model.predict(X_test)
# accuracy_score compare les prédictions (y_pred) à la vérité (y_test)
# et renvoie le pourcentage de bonnes réponses.
print("Accuracy :", accuracy_score(y_test, y_pred))
# classification_report détaille la précision et le rappel, classe par classe.
print(classification_report(y_test, y_pred))

In [ ]:
import joblib
# joblib.dump() sérialise (transforme en fichier binaire) l'objet "model" entraîné.
joblib.dump(model, "model.pkl")
print("Modèle sauvegardé dans model.pkl")

In [ ]:
# ---------- ESSAI 2 : un modèle plus simple ----------
model = RandomForestClassifier(
n_estimators=50, # moins d'arbres qu'à l'essai 1 (100 -> 50)
max_depth=3, # arbres moins profonds qu'à l'essai 1 (5 -> 3)
random_state=42
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy Essai 2 :", accuracy_score(y_test, y_pred))
joblib.dump(model, "model.pkl") # écrase le fichier de l'Essai 1
print("model.pkl écrasé avec l'Essai 2")